# 2048 재합성 검증

후처리를 1024 축소본이 아니라 저장본(2048) 위에서 돌리는 수정(feat/fullres-recompose) 검증.
같은 시드·참조·파라미터의 1024 결과(sweep_0825, seed 42, B2)와 나란히 본다.

In [ ]:
%cd /content
import importlib
import os
import shutil
import sys
import time
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive
from PIL import Image

drive.mount("/content/drive")

REPO = "/content/SalonCutAI"
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q -b feat/fullres-recompose https://github.com/qja0707/SalonCutAI.git {REPO}
!pip install -q diffusers==0.39.0 transformers==5.14.1 peft==0.19.1 accelerate==1.14.0 \
    insightface onnxruntime mediapipe==1.0.0 opencv-contrib-python-headless \
    facexlib torchvision

os.environ["IMAGE_GEN_ENABLED"] = "1"
os.environ["SALON_STORAGE_DIR"] = "/content/storage"
sys.path.insert(0, f"{REPO}/backend")
importlib.invalidate_caches()

from src.ai_engine.image_gen import downloads, loader, pipeline, settings, storage

!cd {REPO} && git log --oneline -1
downloads.ensure_models()
print("missing:", downloads.missing_files())

loader.get_face_app()
loader.get_landmarker()
loader.get_segmenter()
loader.get_codeformer()
loader.get_face_helper()
loader.get_combo3()

import torch
print(f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
SALON = Path("/content/drive/MyDrive/saloncut_data/test_images/salon")
CAND_DIR = Path("/content/drive/MyDrive/saloncut_data/ref_faces/candidates_0825")
SWEEP = Path("/content/drive/MyDrive/saloncut_data/outputs/sweep_0825")
OUT_FR = Path("/content/drive/MyDrive/saloncut_data/outputs/fullres_0825")
OUT_FR.mkdir(parents=True, exist_ok=True)
SEED = 42

shutil.copy(CAND_DIR / "B2.png", settings.REF_FACES_DIR / "ref-B2.png")
opts = SimpleNamespace(
    face=SimpleNamespace(mode="reference", reference=SimpleNamespace(reference_face_id="ref-B2"))
)

for name in ["salon_01_long_wave_brown", "salon_04_long_wave_black"]:
    src = storage.to_stored_size(Image.open(SALON / f"{name}.jpg").convert("RGB"))
    old = Image.open(SWEEP / f"c3_{name}_B2_st0.4_ip0.5.png")

    t = time.time()
    new = pipeline._run_reference_mode(src, opts, SEED)
    dt = time.time() - t
    new.save(OUT_FR / f"c3_{name}_B2_fullres.png")
    print(f"{name}  {dt:.0f}s  old {old.size}  new {new.size}")

    det = loader.get_face_app().get(np.array(src)[:, :, ::-1])
    x1, y1, x2, y2 = det[0].bbox.astype(int)
    fw = x2 - x1
    crops = {
        "hairline": (x1 - fw // 2, y1 - fw // 2, x2 + fw // 2, y1 + fw // 3),
        "jaw": (x1 - fw // 4, y2 - fw // 2, x2 + fw // 4, y2 + fw // 2),
    }
    old_up = old.resize(src.size, Image.LANCZOS)

    fig, axes = plt.subplots(3, 3, figsize=(21, 21))
    for col, (lab, im) in enumerate([("src", src), ("old 1024", old_up), ("new 2048", new)]):
        axes[0, col].imshow(im)
        axes[0, col].set_title(f"{name[6:8]} {lab}", fontsize=13)
        for row, (ck, box) in enumerate(crops.items(), start=1):
            axes[row, col].imshow(im.crop(box))
            axes[row, col].set_title(f"{lab} {ck}", fontsize=12)
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT_FR / f"cmp_{name}_B2.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
import cv2
from src.ai_engine.image_gen import masks

def hair_sharpness(img, hair_mask):
    g = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    m = np.array(hair_mask.resize(img.size)) > 127
    lap = cv2.Laplacian(g, cv2.CV_64F)
    return lap[m].var()

for name in ["salon_01_long_wave_brown", "salon_04_long_wave_black"]:
    src = storage.to_stored_size(Image.open(SALON / f"{name}.jpg").convert("RGB"))
    old = Image.open(SWEEP / f"c3_{name}_B2_st0.4_ip0.5.png").resize(src.size, Image.LANCZOS)
    new = Image.open(OUT_FR / f"c3_{name}_B2_fullres.png")
    hair = masks.build_hair_mask(src, dilate=0)
    print(f"{name[6:8]}  src {hair_sharpness(src, hair):.1f}  old {hair_sharpness(old, hair):.1f}  new {hair_sharpness(new, hair):.1f}")